### Model Comparison

Candidate models x feature sets with cross-validation

In [2]:
# import libraries
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [3]:
# load the cleaned/transformed data and split into train/test sets
df = pd.read_csv("../data/stars_transformed.csv")
y = df["Type"]
X = df.drop(columns=["Type", "Type_Name"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)     # test set stays untouched until notebook 05

(192, 17) (48, 17)


In [4]:
# define feature sets for model comparison (the experiment axis to compare the effect of adding more features)
color_cols = [c for c in X.columns if c.startswith("color_")]
base = ["log_Temperature", "log_L"]

FEATURE_SETS = {
    "A: log T + log L":            base,
    "B: + log R":                  base + ["log_R"],
    "C: + A_M (numeric only)":     base + ["log_R", "A_M"],
    "D: C + spectral":             base + ["log_R", "A_M", "Spectral_Ordinal"],
    "E: D + color":                base + ["log_R", "A_M", "Spectral_Ordinal"] + color_cols,
}

In [5]:
# define candidate models for comparison
MODELS = {
    "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "KNN (k=5)":           make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(300, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

In [6]:
# define cross-validation strategy and scoring metrics
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {"acc": "accuracy", "f1": "f1_macro", "logloss": "neg_log_loss"}

rows = []
for fs_name, cols in FEATURE_SETS.items():
    for m_name, model in MODELS.items():
        r = cross_validate(model, X_train[cols], y_train, cv=cv, scoring=scoring)
        rows.append({"feature set": fs_name, "model": m_name,
                     "accuracy": r["test_acc"].mean(), "acc_std": r["test_acc"].std(),
                     "macro_f1": r["test_f1"].mean(),
                     "log_loss": -r["test_logloss"].mean()})
results = pd.DataFrame(rows)
results.pivot(index="model", columns="feature set", values="accuracy").round(3)

c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

feature set,A: log T + log L,B: + log R,C: + A_M (numeric only),D: C + spectral,E: D + color
model,,,,,
Baseline (majority),0.156,0.156,0.156,0.156,0.156
Decision Tree,0.830,0.943,0.998,0.989,0.997
Gradient Boosting,0.789,0.952,0.989,0.985,0.984
KNN (k=5),0.787,0.950,0.987,0.976,0.921
Logistic Regression,0.792,0.940,0.970,0.962,0.961
Random Forest,0.806,0.959,1.000,1.000,0.998
SVM (RBF),0.791,0.954,0.982,0.971,0.907


In [7]:
# sort results by accuracy (descending) and log loss (ascending)
results.sort_values(["accuracy", "log_loss"], ascending=[False, True]).head(15).round(3)

,feature set,model,accuracy,acc_std,macro_f1,log_loss
19,C: + A_M (numeric only),Random Forest,1.000,0.000,1.000,0.022
26,D: C + spectral,Random Forest,1.000,0.000,1.000,0.035
33,E: D + color,Random Forest,0.998,0.007,0.998,0.063
18,C: + A_M (numeric only),Decision Tree,0.998,0.007,0.998,0.076
32,E: D + color,Decision Tree,0.997,0.009,0.997,0.114
20,C: + A_M (numeric only),Gradient Boosting,0.989,0.015,0.988,0.024
25,D: C + spectral,Decision Tree,0.989,0.013,0.988,0.414
16,C: + A_M (numeric only),KNN (k=5),0.987,0.020,0.987,0.072
27,D: C + spectral,Gradient Boosting,0.985,0.015,0.985,0.026
34,E: D + color,Gradient Boosting,0.984,0.017,0.984,0.026
